# Создание аугментаций

Можно воспользоваться этим кодом для получения аугментаций, но только в синхронном формате. Для асинхронного формата необходимо запустить скрипт `create_augmentation.py`. В этом скрипте идентичный код.

In [1]:
from autointent import load_dataset
import json
import pandas as pd
import os


def create_subset(dataset_name: str, max_utterances_per_intent: int = 10):
    output_file = f"datasets/{dataset_name.split('/')[-1]}.json"
    output_file_subset = output_file.replace(".json", f"_subset_{max_utterances_per_intent}.json")

    if os.path.isfile(output_file_subset):
        return output_file, output_file_subset
    
    dataset = load_dataset(dataset_name)
    dataset.to_json(output_file)
    
    with open(output_file, 'r') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data["train"])
    sampled_df = df.groupby("label").head(max_utterances_per_intent)
    data["train"] = sampled_df.to_dict(orient="records")

    with open(output_file_subset, 'w') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    
    return output_file, output_file_subset

/home/darinka/AutoIntent/.venv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
datasets = [
    "AutoIntent/banking77_ru",
    "AutoIntent/clinc150_ru",
    "AutoIntent/banking77",
    "AutoIntent/snips",
    "AutoIntent/snips_ru"
]

datasets_subset = []

for dataset_name in datasets:
    _, output_file_subset = create_subset(dataset_name, max_utterances_per_intent=10)
    datasets_subset.append(output_file_subset)

In [3]:
from autointent.generation.utterances.evolution.evolver import UtteranceEvolver
from autointent.generation.utterances.generator import Generator
from autointent.generation.utterances.evolution.chat_templates import (
    AbstractEvolution,
    ConcreteEvolution,
    FormalEvolution,
    FunnyEvolution,
    GoofyEvolution,
    InformalEvolution,
    ReasoningEvolution,
)
import os

os.environ["OPENAI_MODEL_NAME"] = 'Qwen/Qwen2.5-7B-Instruct-AWQ'
os.environ["OPENAI_BASE_URL"] = 'http://localhost:8000/v1'
os.environ["OPENAI_API_KEY"] = 'sth'

In [4]:
datasets_subset

['datasets/banking77_ru_subset_10.json',
 'datasets/clinc150_ru_subset_10.json',
 'datasets/banking77_subset_10.json',
 'datasets/snips_subset_10.json',
 'datasets/snips_ru_subset_10.json']

In [ ]:
evolutions = [
        AbstractEvolution(),
        ConcreteEvolution(),
        FormalEvolution(),
        FunnyEvolution(),
        GoofyEvolution(),
        InformalEvolution(),
        ReasoningEvolution()
]
seed = 42
split = "train"
n_evolutions = 10
datasets_subset_aug = []


for dataset_subset in datasets_subset:
    print(dataset_subset)
    dataset = load_dataset(dataset_subset)
    n_before = len(dataset[split])
    output_path = dataset_subset.replace(".json", "_augmented.json") 
    datasets_subset_aug.append(output_path)

    generator = UtteranceEvolver(Generator(), evolutions, seed, async_mode=False)
    _ = generator.augment(dataset, split_name=split, n_evolutions=n_evolutions)

    dataset.to_json(output_path)

# Проведение экспериментов

In [7]:
datasets = [
    "AutoIntent/banking77_ru",
    "AutoIntent/clinc150_ru",
    "AutoIntent/banking77",
    "AutoIntent/snips",
    "AutoIntent/snips_ru"
]
max_utterances_per_intent = 10

datasets_subset = []
datasets_subset_aug = []

for dataset_name in datasets:
    base_name = dataset_name.split()[-1]
    dataset_subset = f"datasets/{base_name}_subset_{max_utterances_per_intent}.json"
    dataset_subset_aug = dataset_subset.replace(".json", "_augmented.json")

    datasets_subset.append(dataset_subset)
    datasets_subset_aug.append(dataset_subset_aug)

In [9]:
datasets_subset[0], datasets_subset_aug[0]

('datasets/AutoIntent/banking77_ru_subset_10.json',
 'datasets/AutoIntent/banking77_ru_subset_10_augmented.json')

In [2]:
search_space = [
    {
        "node_type": "embedding",
        "target_metric": "retrieval_hit_rate",
        "search_space": [
            {
                "module_name": "retrieval",
                "k": [5, 10],
                "embedder_name": [
                    "avsolatorio/GIST-small-Embedding-v0",
                    "infgrad/stella-base-en-v2",
                    "sentence-transformers/all-MiniLM-L6-v2",
                    "BAAI/bge-reranker-base"
                ],
            }
        ],
    },
    {
        "node_type": "scoring",
        "target_metric": "scoring_roc_auc",
        "metrics": ["scoring_accuracy"],
        "search_space": [
            {"module_name": "linear"}
        ],
    },
    {
        "node_type": "decision",
        "target_metric": "decision_accuracy",
        "search_space": [
            {"module_name": "threshold", "thresh": [0.5]},
            {"module_name": "argmax"},
            {"module_name": "tunable"},
            {"module_name": "jinoos"}
        ],
    },
]

In [3]:
from autointent import Pipeline
from autointent.configs import LoggingConfig, EmbedderConfig
from pathlib import Path

log_config = LoggingConfig(
    report_to=["wandb"], clear_ram=True, dump_modules=True)
emb_config = EmbedderConfig(batch_size=16, device="cuda")


In [4]:
import torch
import wandb

datasets = [
    "/home/darinka/AutoIntent/banking77_ru_subset_aug.json",
    "/home/darinka/AutoIntent/banking77_ru_subset.json",
    "/home/darinka/AutoIntent/banking77_subset_aug.json",
    "/home/darinka/AutoIntent/banking77_subset.json",
    "/home/darinka/AutoIntent/clinc150_subset_aug.json",
    "/home/darinka/AutoIntent/clinc150_subset.json"
]
for dataset in datasets:
    log_config.run_name = f"{dataset}_multiple_metrics"
    dataset = Dataset.from_json(dataset)
    pipeline_optimizer = Pipeline.from_search_space(search_space)
    pipeline_optimizer.set_config(log_config)
    pipeline_optimizer.set_config(emb_config)
    pipeline_optimizer.fit(dataset)
    torch.cuda.empty_cache()
    wandb.finish()

Filter:   0%|          | 0/2310 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1155 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1155 [00:00<?, ? examples/s]

wandb: Currently logged in as: darinka. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


retrieval_hit_rate,▁
retrieval_hit_rate,0.87446


...

Found unexpected child /home/darinka/AutoIntent/banking77_ru_subset_aug.json_multiple_metrics/modules_dumps/NodeType.embedding/retrieval/comb_5/data.json
Found unexpected child /home/darinka/AutoIntent/banking77_ru_subset_aug.json_multiple_metrics/modules_dumps/NodeType.embedding/retrieval/comb_5/metadata.json
No sentence-transformers model found with name infgrad/stella-base-en-v2. Creating a new one with mean pooling.
No sentence-transformers model found with name infgrad/stella-base-en-v2. Creating a new one with mean pooling.
/home/darinka/AutoIntent/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


decision_accuracy,▁
decision_f1,▁
decision_precision,▁
decision_recall,▁
decision_roc_auc,▁
decision_accuracy,0.27565
decision_f1,0.25466
decision_precision,0.3144
decision_recall,0.27565
decision_roc_auc,0.63306


Filter:   0%|          | 0/385 [00:00<?, ? examples/s]

Filter:   0%|          | 0/192 [00:00<?, ? examples/s]

ValueError: The test_size = 39 should be greater or equal to the number of classes = 77